In [5]:
import os
import h5py
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDClassifier  # Stochastic Gradient Descent for streaming
from sklearn.metrics import accuracy_score, roc_auc_score, precision_recall_fscore_support

# Define the TCGA Folder Path
base_path = os.path.expanduser("~/Downloads/tcga-selected")  # Update if needed
luad_path = os.path.join(base_path, "LUAD")
lusc_path = os.path.join(base_path, "LUSC")

# Function to get all .h5 files recursively
def get_h5_files(root_folder):
    h5_files = []
    for subdir, _, files in os.walk(root_folder):
        for file in files:
            if file.endswith(".h5"):
                h5_files.append(os.path.join(subdir, file))
    return h5_files

# Get LUAD and LUSC .h5 files
luad_files = get_h5_files(luad_path)
lusc_files = get_h5_files(lusc_path)

# Split into train-test
train_luad, test_luad = train_test_split(luad_files, test_size=0.2, random_state=42)
train_lusc, test_lusc = train_test_split(lusc_files, test_size=0.2, random_state=42)
train_files = train_luad + train_lusc
test_files = test_luad + test_lusc

# Function to load features one file at a time
def stream_features(file_list):
    for file_path in file_list:
        with h5py.File(file_path, "r") as h5_file:
            if "features" in h5_file:
                features = h5_file["features"][:]
                label = 0 if "LUAD" in file_path else 1
                labels = np.full(features.shape[0], label)
                yield features, labels

# Initialize online training model & scaler
scaler = StandardScaler()
sgd_clf = SGDClassifier(loss="log_loss", max_iter=1000, tol=1e-3)


# **📌 Step 1: Train Model Incrementally**
for X_train, y_train in stream_features(train_files):
    X_train_scaled = scaler.partial_fit(X_train).transform(X_train)  # Scale incrementally
    sgd_clf.partial_fit(X_train_scaled, y_train, classes=np.array([0, 1]))  # Incremental training

# **📌 Step 2: Evaluate Model on Test Set**
y_true, y_pred, y_prob = [], [], []
for X_test, y_test in stream_features(test_files):
    X_test_scaled = scaler.transform(X_test)  # Use fitted scaler
    y_pred.extend(sgd_clf.predict(X_test_scaled))
    y_prob.extend(sgd_clf.decision_function(X_test_scaled))  # Probability estimation
    y_true.extend(y_test)

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)

# **📌 Step 3: Evaluate Model Performance**
accuracy = accuracy_score(y_true, y_pred)
auroc = roc_auc_score(y_true, y_prob)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary")

# **📌 Step 4: Print Results**
print(f"✅ Logistic Regression Model (Streaming) Trained Successfully!")
print(f"🔹 Accuracy: {accuracy:.4f}")
print(f"🔹 AUROC Score: {auroc:.4f}")
print(f"🔹 Precision: {precision:.4f}")
print(f"🔹 Recall: {recall:.4f}")
print(f"🔹 F1 Score: {f1:.4f}")


✅ Logistic Regression Model (Streaming) Trained Successfully!
🔹 Accuracy: 0.7244
🔹 AUROC Score: 0.8156
🔹 Precision: 0.7586
🔹 Recall: 0.7011
🔹 F1 Score: 0.7287
